# 02 · Reference Models, LSTM Training & Backtests

Step 1 of the exercise — **demand forecasting with time-series, regression and LSTM models**.

This notebook
1. tunes the ridge penalty on the validation window,
2. proves the hand-written ridge and ETS recursions match scikit-learn / statsmodels (they are the
   specification the browser models are tested against),
3. trains the global **LSTM** offline and exports its weights for TensorFlow.js,
4. backtests every model with rolling origins on the validation window and on the untouched holdout,
5. writes `dashboard/public/data/model_config.json` (penalty, ensemble weights, interval offsets, leaderboards).

**Split** — train: 2022-01-01 → 2023-09-02 · validation: → 2023-12-01 · holdout: 2023-12-02 → 2024-01-30.
**No leakage:** price columns are never used; future epidemic status is held at its last observed value.

In [1]:
import os
import sys
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))

import json
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import Ridge

import reference_models as rm
from inventory_data import DATA_OUT, EXOG_FEATURES, build_panel, exog_matrix, load_raw, validate

pd.set_option("display.width", 160)
df = load_raw(ROOT / "sales_data.csv")
validate(df)
panel = build_panel(df)
split = panel.split
val_origins = rm.rolling_origins(split.train_end, split.val_end)
holdout_origins = rm.rolling_origins(split.val_end, split.holdout_end)
print("validation origins:", [panel.dates[o].date().isoformat() for o in val_origins])
print("holdout origins:   ", [panel.dates[o].date().isoformat() for o in holdout_origins])

validation origins: ['2023-09-02', '2023-09-09', '2023-09-16', '2023-09-23', '2023-09-30', '2023-10-07', '2023-10-14', '2023-10-21', '2023-10-28', '2023-11-04', '2023-11-11']
holdout origins:    ['2023-12-01', '2023-12-08', '2023-12-15', '2023-12-22', '2023-12-29', '2024-01-05', '2024-01-12']


## 1 · Ridge penalty (validation window)
Per-SKU ridge regression on lags (1, 7, 14), rolling means (7, 28), promotion, epidemic, weather and yearly
Fourier terms; standardised features, unpenalised intercept, recursive 14-day forecasts.

In [2]:
alpha_grid = [0.1, 1.0, 10.0, 100.0, 1000.0]
alpha_scores = {}
for a in alpha_grid:
    bt = rm.run_backtest(panel, val_origins, {"ridge": rm.make_ridge_forecaster(a)})
    alpha_scores[a] = {"WAPE h=1": rm.metrics(bt[bt.h == 1].y, bt[bt.h == 1].f)["WAPE"],
                       "WAPE h≤14": rm.metrics(bt.y, bt.f)["WAPE"]}
alpha_table = pd.DataFrame(alpha_scores).T
RIDGE_ALPHA = float(alpha_table["WAPE h≤14"].idxmin())
print("chosen alpha =", RIDGE_ALPHA)
alpha_table.round(4)

chosen alpha = 100.0


,WAPE h=1,WAPE h≤14
0.1,0.2524,0.2955
1.0,0.2523,0.2954
10.0,0.2517,0.2946
100.0,0.2511,0.2901
1000.0,0.2775,0.3003


## 2 · Parity checks of the reference recursions
The browser re-implements these exact recursions, so first confirm they agree with the standard libraries.

In [3]:
s, origin = panel.skus.index("S001-P0001"), split.train_end
exog = exog_matrix(panel, s)
y = panel.demand[s].astype(float)
fit = rm.fit_ridge(y, exog, origin, RIDGE_ALPHA)
X = np.array([rm.ridge_row(y, exog, t) for t in range(rm.MIN_HISTORY, origin + 1)])
Z = (X - fit.mean) / fit.scale
sk = Ridge(alpha=RIDGE_ALPHA).fit(Z, y[rm.MIN_HISTORY:origin + 1])
print("ridge  max |coef - sklearn| =", np.abs(sk.coef_ - fit.coef).max(), "| intercept diff =", abs(sk.intercept_ - fit.intercept))

import warnings

from statsmodels.tsa.holtwinters import ExponentialSmoothing

ets = rm.fit_ets(y[: origin + 1])
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sm = ExponentialSmoothing(y[: origin + 1], trend="add", damped_trend=True, initialization_method="known",
                              initial_level=ets.l0, initial_trend=ets.b0, bounds=rm.ETS_BOUNDS).fit()
p = sm.params
fitted, level, trend = rm.ets_filter(y[: origin + 1], p["smoothing_level"], p["smoothing_trend"],
                                     p["damping_trend"], ets.l0, ets.b0)
forecast = rm.ets_forecast_from_state(level, trend, p["damping_trend"], 14)
print("ETS    max |fitted - statsmodels| =", np.abs(fitted - sm.fittedvalues).max(),
      "| max |forecast - statsmodels| =", np.abs(forecast - sm.forecast(14)).max())
print(f"ETS selected for S001-P0001: {ets.kind} (alpha={ets.alpha:.4f}, beta={ets.beta}, phi={ets.phi}), AIC={ets.aic:.1f}")

ridge  max |coef - sklearn| = 1.2434497875801753e-14 | intercept diff = 0.0


ETS    max |fitted - statsmodels| = 0.0 | max |forecast - statsmodels| = 0.0
ETS selected for S001-P0001: SES (alpha=0.1328, beta=None, phi=None), AIC=4283.2


## 3 · Global LSTM (offline training)
One model for all 100 SKUs. Input: the last 28 days of scaled demand + exogenous drivers, plus the known
exogenous drivers of the next 14 days. Output: 14 days of scaled demand (scale = SKU mean over the training
period). Early stopping uses the **last 60 days of the training period**, so the validation window stays
untouched for the leaderboard.

In [4]:
import keras

keras.utils.set_random_seed(42)
scales = rm.lstm_scales(panel)
fit_origins = range(rm.LSTM_WINDOW - 1, split.train_end - rm.HORIZON - 60 + 1)
inner_origins = range(fit_origins.stop, split.train_end - rm.HORIZON + 1)
Xe, Xf, Yt = rm.lstm_training_set(panel, fit_origins, scales)
Ve, Vf, Vt = rm.lstm_training_set(panel, inner_origins, scales)
print("training windows", Xe.shape, Xf.shape, "| early-stopping windows", Ve.shape)

model = rm.build_lstm()
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
t0 = time.time()
history = model.fit(
    [Xe, Xf], Yt, validation_data=([Ve, Vf], Vt), epochs=60, batch_size=256, verbose=0,
    callbacks=[keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)],
)
best_epoch = int(np.argmin(history.history["val_loss"])) + 1
print(f"trained {len(history.history['loss'])} epochs in {time.time() - t0:.0f}s; best epoch {best_epoch}, "
      f"val MSE {min(history.history['val_loss']):.4f} (scaled units)")
model.summary()

training windows (50900, 28, 10) (50900, 126) | early-stopping windows (6000, 28, 10)


trained 11 epochs in 18s; best epoch 5, val MSE 0.1427 (scaled units)


Model: "lstm_forecaster"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder             │ (None, 28, 10)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 32)        │      5,504 │ encoder[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ future (InputLayer) │ (None, 126)       │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat              │ (None, 158)       │          0 │ lstm[0][0],       │
│ (Concatenate)       │                   │            │ future[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_hidden        │ (None, 64)        │     10,176 │ concat[0][0]      │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_out (Dense)   │ (None, 14)        │        910 │ dense_hidden[0][… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 49,772 (194.43 KB)

 Trainable params: 16,590 (64.80 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 33,182 (129.62 KB)

In [5]:
weights_payload = rm.export_lstm_weights(model)
weights_payload["trainedThroughIdx"] = int(split.train_end)
weights_payload["trainedThrough"] = panel.dates[split.train_end].date().isoformat()
(DATA_OUT / "lstm").mkdir(parents=True, exist_ok=True)
(DATA_OUT / "lstm" / "weights.json").write_text(json.dumps(weights_payload), encoding="utf-8")

reloaded = rm.build_lstm()
rm.load_lstm_weights(reloaded, json.loads((DATA_OUT / "lstm" / "weights.json").read_text()))
diff = np.abs(model.predict([Ve[:256], Vf[:256]], verbose=0) - reloaded.predict([Ve[:256], Vf[:256]], verbose=0)).max()
print(f"exported {sum(w.size for w in model.get_weights())} parameters; max |original - reloaded| = {diff:.2e}")

exported 16590 parameters; max |original - reloaded| = 0.00e+00


## 4 · Validation backtest (rolling origins, refit at every origin)

In [6]:
forecasters = {
    "Static mean (baseline)": rm.static_mean_forecaster,
    "Naive": rm.naive_forecaster,
    "MA-28": rm.ma28_forecaster,
    "ETS": rm.ets_forecaster,
    "Ridge": rm.make_ridge_forecaster(RIDGE_ALPHA),
    "LSTM": rm.make_lstm_forecaster(model, scales),
}
t0 = time.time()
bt_val = rm.run_backtest(panel, val_origins, forecasters)
print(f"validation backtest: {len(bt_val):,} forecasts in {time.time() - t0:.0f}s")


def with_ensemble(bt: pd.DataFrame, weights: dict[str, float]) -> pd.DataFrame:
    keys = ["sku", "origin", "h"]
    members = bt[bt.model.isin(weights)].pivot_table(index=keys, columns="model", values="f")
    ens = (members[list(weights)] * pd.Series(weights)).sum(axis=1).rename("f").reset_index()
    base = bt[bt.model == next(iter(weights))][keys + ["y", "mase_scale"]]
    return pd.concat([bt, base.merge(ens, on=keys).assign(model="Ensemble")], ignore_index=True)


member_wape = {m: rm.metrics(g.y, g.f)["WAPE"] for m, g in bt_val[bt_val.model.isin(["ETS", "Ridge", "LSTM"])].groupby("model")}
inv = {m: 1 / w for m, w in member_wape.items()}
ENSEMBLE_WEIGHTS = {m: inv[m] / sum(inv.values()) for m in ["ETS", "Ridge", "LSTM"]}
print("ensemble weights (inverse validation WAPE):", {k: round(v, 3) for k, v in ENSEMBLE_WEIGHTS.items()})
bt_val = with_ensemble(bt_val, ENSEMBLE_WEIGHTS)


def boards(bt):
    return {f"h{h}": rm.leaderboard(bt[bt.h <= h] if h > 1 else bt[bt.h == 1]) for h in (1, 7, 14)}


val_boards = boards(bt_val)
val_boards["h14"].round(4)

validation backtest: 92,400 forecasts in 23s
ensemble weights (inverse validation WAPE): {'ETS': 0.302, 'Ridge': 0.351, 'LSTM': 0.347}


,MAE,RMSE,WAPE,MAPE,sMAPE,Bias,MASE
Ridge,28.4701,36.7865,0.2901,0.4137,0.3157,-0.0020,0.6942
LSTM,28.8405,36.7845,0.2939,0.4302,0.3195,-0.0045,0.7040
Ensemble,28.8488,36.8525,0.2940,0.4313,0.3191,0.0023,0.7047
Static mean (baseline),32.6732,40.5740,0.3329,0.5442,0.3529,0.0702,0.7983
ETS,33.0441,41.7611,0.3367,0.4977,0.3604,0.0151,0.8086
MA-28,34.7556,43.1720,0.3542,0.5494,0.3760,0.0260,0.8522
Naive,40.3043,51.9736,0.4107,0.5238,0.4342,-0.0163,0.9802


In [7]:
by_h = bt_val.assign(ae=(bt_val.f - bt_val.y).abs()).groupby(["model", "h"]).apply(
    lambda g: g.ae.sum() / g.y.sum(), include_groups=False).unstack("model")
by_h.round(3)

model,ETS,Ensemble,LSTM,MA-28,Naive,Ridge,Static mean (baseline)
h,,,,,,,
1,0.298,0.262,0.274,0.338,0.380,0.251,0.338
2,0.300,0.255,0.261,0.328,0.397,0.243,0.333
3,0.306,0.269,0.281,0.342,0.397,0.257,0.348
4,0.326,0.275,0.271,0.342,0.388,0.270,0.317
5,0.326,0.283,0.280,0.346,0.395,0.282,0.319
6,0.330,0.282,0.277,0.347,0.403,0.277,0.326
7,0.357,0.305,0.300,0.380,0.421,0.300,0.346
8,0.337,0.295,0.297,0.365,0.408,0.293,0.340
9,0.333,0.292,0.292,0.349,0.418,0.292,0.338


### Is the best model significantly better than the runner-up? (Diebold–Mariano, squared loss, h = 1)

In [8]:
def diebold_mariano(bt, a, b, h=1):
    keys = ["sku", "origin", "h"]
    sub = bt[bt.h == h]
    ea = sub[sub.model == a].set_index(keys)
    eb = sub[sub.model == b].set_index(keys).loc[ea.index]
    d = (ea.f - ea.y) ** 2 - (eb.f - eb.y) ** 2
    stat = d.mean() / np.sqrt(d.var(ddof=1) / len(d))
    return stat, 2 * (1 - stats.norm.cdf(abs(stat)))


ranked = val_boards["h1"].index.tolist()
stat, pval = diebold_mariano(bt_val, ranked[0], ranked[1])
print(f"h=1: {ranked[0]} vs {ranked[1]}: DM = {stat:.2f}, p = {pval:.4f}")
stat, pval = diebold_mariano(bt_val, ranked[0], "Static mean (baseline)")
print(f"h=1: {ranked[0]} vs Static mean (baseline): DM = {stat:.2f}, p = {pval:.2e}")

h=1: Ridge vs Ensemble: DM = -4.88, p = 0.0000
h=1: Ridge vs Static mean (baseline): DM = -13.51, p = 0.00e+00


## 5 · Holdout backtest (data never used for tuning)
The holdout includes the 13-day epidemic of 27 Dec 2023 – 8 Jan 2024; in the dashboard this period is replayed
day by day as "new data arriving".

In [9]:
t0 = time.time()
bt_hold = with_ensemble(rm.run_backtest(panel, holdout_origins, forecasters), ENSEMBLE_WEIGHTS)
print(f"holdout backtest: {len(bt_hold):,} forecasts in {time.time() - t0:.0f}s")
hold_boards = boards(bt_hold)
pd.concat({"h=1": hold_boards["h1"][["WAPE", "MAE", "MASE", "Bias"]],
           "h≤14": hold_boards["h14"][["WAPE", "MAE", "MASE", "Bias"]]}, axis=1).round(4)

holdout backtest: 68,600 forecasts in 15s


h=1                             h≤14                         
                          WAPE      MAE    MASE    Bias    WAPE      MAE    MASE    Bias
Ridge                   0.2659  26.1932  0.6378  0.0038  0.3292  32.6839  0.7989  0.0114
Ensemble                0.2775  27.3400  0.6661 -0.0117  0.3332  33.0817  0.8091 -0.0039
LSTM                    0.2786  27.4509  0.6654 -0.0492  0.3452  34.2695  0.8355 -0.0276
ETS                     0.3387  33.3661  0.8153  0.0132  0.3671  36.4452  0.8938  0.0055
Static mean (baseline)  0.3698  36.4301  0.8970  0.0614  0.3524  34.9848  0.8547  0.0533
MA-28                   0.3720  36.6464  0.9056  0.0616  0.3667  36.4080  0.8929  0.0536
Naive                   0.4271  42.0743  1.0175  0.0233  0.4605  45.7169  1.1115  0.0156

## 6 · Prediction intervals (conformal-style) and their holdout coverage
Interval offsets are the empirical quantiles of validation errors per model and horizon. Coverage is checked
on the holdout, which the offsets never saw.

In [10]:
intervals = rm.residual_quantiles(bt_val)


def coverage(bt, q, lo, hi):
    out = {}
    for m, g in bt.groupby("model"):
        lo_off = np.array(q[m][lo])[g.h - 1]
        hi_off = np.array(q[m][hi])[g.h - 1]
        out[m] = float(np.mean((g.y >= g.f + lo_off) & (g.y <= g.f + hi_off)))
    return out


cov = pd.DataFrame({"80% band": coverage(bt_hold, intervals, "q100", "q900"),
                    "95% band": coverage(bt_hold, intervals, "q025", "q975")})
cov.round(3)

,80% band,95% band
ETS,0.753,0.918
Ensemble,0.729,0.911
LSTM,0.707,0.902
MA-28,0.772,0.933
Naive,0.745,0.917
Ridge,0.734,0.917
Static mean (baseline),0.755,0.929


## 7 · Export the model configuration for the dashboard

In [11]:
def board_json(b):
    return {k: v.round(6).to_dict(orient="index") for k, v in b.items()}


config = {
    "generatedAt": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "horizon": rm.HORIZON,
    "minHistory": rm.MIN_HISTORY,
    "exogFeatures": EXOG_FEATURES,
    "futureEpidemic": "persistence (last observed value)",
    "ridge": {"alpha": RIDGE_ALPHA, "features": rm.RIDGE_FEATURES,
              "alphaGrid": {str(k): v for k, v in alpha_table.round(6).to_dict(orient="index").items()}},
    "ets": {"bounds": {k: list(v) for k, v in rm.ETS_BOUNDS.items()},
            "initialState": "level = mean(y[0:7]); trend = (mean(y[7:14]) - mean(y[0:7])) / 7",
            "selection": "AIC = n*ln(SSE/n) + 2k with k=1 (SES) or k=3 (damped Holt)"},
    "lstm": {"weightsFile": "lstm/weights.json", "window": rm.LSTM_WINDOW,
             "trainedThroughIdx": int(split.train_end), "epochs": len(history.history["loss"]), "bestEpoch": best_epoch,
             "loss": [round(float(v), 6) for v in history.history["loss"]],
             "valLoss": [round(float(v), 6) for v in history.history["val_loss"]],
             "scales": {sku: round(float(sc), 6) for sku, sc in zip(panel.skus, scales)}},
    "ensemble": {"members": list(ENSEMBLE_WEIGHTS), "weights": {k: round(v, 6) for k, v in ENSEMBLE_WEIGHTS.items()},
                 "rule": "inverse validation WAPE (h<=14)"},
    "intervals": {"source": "validation residuals y - f, by model and horizon", "byModel": intervals},
    "backtest": {
        "validation": {"origins": [panel.dates[o].date().isoformat() for o in val_origins], "leaderboard": board_json(val_boards)},
        "holdout": {"origins": [panel.dates[o].date().isoformat() for o in holdout_origins], "leaderboard": board_json(hold_boards),
                    "intervalCoverage": cov.round(4).to_dict(orient="index")},
    },
}
(DATA_OUT / "model_config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")
print("wrote", (DATA_OUT / "model_config.json").relative_to(ROOT))

wrote dashboard\public\data\model_config.json
